In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
races_schema = StructType([
    StructField("raceId", IntegerType(), False),
    StructField("year", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("circuitId", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("date", DateType(), True),
    StructField("time", StringType(), True),
    StructField("url", StringType(), True)
])
races_df = spark.read.csv(
    f"{raw_folder_path}/races.csv",
    header=True,
    schema=races_schema
)
display(races_df)
races_df.printSchema()
races_df.describe().show()

In [0]:
from pyspark.sql.functions import current_timestamp, to_timestamp, concat, col, lit, try_to_timestamp

In [0]:
races_added_df = races_df \
    .withColumn("race_timestamp", try_to_timestamp(concat(col("date"), lit(" "), col("time")), lit("yyyy-MM-dd HH:mm:ss"))) \
    .withColumn("ingestion_date", current_timestamp())

display(races_added_df)

In [0]:
races_selected_df = races_added_df.select("raceId", "year", "round", "circuitId", "name", "race_timestamp", "ingestion_date")
races_final = races_selected_df.withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("year", "race_year") \
    .withColumnRenamed("circuitId", "circuit_id")
display(races_final)

In [0]:
races_final.write.mode("overwrite").partitionBy("race_year").parquet(f"{processed_folder_path}/races")

In [0]:
df = spark.read.parquet(f"{processed_folder_path}/races")
display(df)

In [0]:
dbutils.notebook.exit("Success")